In [2]:
import numpy as np
import matplotlib.pyplot as plt

In [3]:
HBAR = 1.054571817e-34
ELECTRON_MASS = 9.1093837e-31
EV_TO_JOULE = 1.602176634e-19
NM_TO_METRE = 1e-9

In [4]:
def calculate_well_strength(depth_ev,width_nm,mass_ratio=1.0):
    if depth_ev <= 0:
        raise ValueError("Well depth must be greater than zero.")
    if width_nm <= 0:
        raise ValueError("Well width must be greater than zero.")
    if mass_ratio <= 0:
        raise ValueError("Mass ratio must be greater than zero.")

    depth_joule = depth_ev*EV_TO_JOULE
    width_metre = width_nm*NM_TO_METRE
    half_width = width_metre/2
    particle_mass = mass_ratio*ELECTRON_MASS
    z0 = (half_width*np.sqrt(2*particle_mass*depth_joule)/HBAR)

    return z0,half_width

In [5]:
def even_transcendental(z,z0):
  z_arr = np.asarray(z,dtype=float)

  if z0 <= 0:
    raise ValueError("z0 must be greater than zero.")
  if np.any(z_arr <= 0) or np.any(z_arr >= z0):
    raise ValueError("All values of z must lie in interval (0,z0)")

  sqrt_term = np.sqrt(np.clip(z0**2- z_arr**2,0.0,None))
  f_even    = z_arr*np.tan(z_arr) - sqrt_term

  return f_even

In [6]:
def odd_transcendental(z,z0):
  z_arr = np.asarray(z,dtype=float)

  if z0 <= 0:
    raise ValueError("z0 must be greater than zero.")
  if np.any(z_arr <= 0) or np.any(z_arr >= z0):
    raise ValueError("All values of z must lie in interval (0,z0)")

  sqrt_term = np.sqrt(np.clip(z0**2- z_arr**2,0.0,None))
  f_odd   = -z_arr/np.tan(z_arr) - sqrt_term

  return f_odd

In [7]:
def bisection_method(function,left,right,z0,tolerance=1e-10,max_iterations=1000):
    error = abs(right - left)
    itr   = 0
    f_left  = function(left,z0)
    f_right = function(right,z0)

    if f_left*f_right>0:
        raise ValueError("Root doesn't exist in given interval")

    elif abs(f_left) < tolerance:
        return left

    elif abs(f_right) < tolerance:
        return right

    else:
        while error > tolerance and itr < max_iterations:
            c   = (left+right)/2
            f_c = function(c,z0)

            if abs(f_c) < tolerance:
                return c

            elif f_left*f_c < 0:
                right = c
                f_right = f_c
                error = abs(right - left)

            else:
                left = c
                f_left = f_c
                error = abs(right - left)
            itr += 1

    if error <= tolerance:
      return (left + right) / 2

    raise RuntimeError("Bisection method did not converge within max_iterations.")

In [20]:
def intervals(z0):
    step = np.pi/2
    z_val = np.arange(0,z0,step)

    z_even = []
    z_odd  = []
    for i in range(len(z_val) - 1):
        tele = (z_val[i],z_val[i+1])
        if i%2 == 0:
            z_even.append(tele)
        else:
            z_odd.append(tele)
    z_odd.append((z_val[-1],z0))

    return z_even,z_odd


In [27]:
def sign_check(even_intervals,odd_intervals,z0):
    even_valid_intervals = []
    odd_valid_intervals  = []
    epsilon = 1e-10

    for i in range(len(even_intervals)):
        ele_even = even_intervals[i]

        left_e  = ele_even[0] + epsilon
        right_e = ele_even[1] - epsilon

        left_even  = even_transcendental(left_e,z0)
        right_even = even_transcendental(right_e,z0)

        if left_even*right_even < 0:
            even_valid_intervals.append((left_e,right_e))

    for j in range(len(odd_intervals)):
        ele_odd  = odd_intervals[j]

        left_o  = ele_odd[0] + epsilon
        right_o = ele_odd[1] - epsilon

        left_odd  = odd_transcendental(left_o,z0)
        right_odd = odd_transcendental(right_o,z0)

        if left_odd*right_odd < 0:
            odd_valid_intervals.append((left_o,right_o))
            
    return even_valid_intervals,odd_valid_intervals



In [31]:
def finding_roots(even_valid_intervals,odd_valid_intervals,z0):
    even_roots = []
    odd_roots  = []

    for i in range(len(even_valid_intervals)):
        ele   = even_valid_intervals[i]    
        left  = ele[0]
        right = ele[1] 
        root  = bisection_method(even_transcendental,left,right,z0)
        even_roots.append(root)

    for i in range(len(odd_valid_intervals)):
        ele   = odd_valid_intervals[i]    
        left  = ele[0]
        right = ele[1] 
        root  = bisection_method(odd_transcendental,left,right,z0)
        odd_roots.append(root)

    return even_roots,odd_roots

In [8]:
z0, a = calculate_well_strength(depth_ev=10,width_nm=1,mass_ratio=1)

print("z0 =", z0)
print("Half-width =", a, "m")

print(even_transcendental(1.0, z0))

z_test = np.array([0.5, 1.0, 1.4])
print(even_transcendental(z_test, z0))

epsilon = 1e-8

first_odd_root = bisection_method(odd_transcendental,np.pi / 2 + epsilon,np.pi - epsilon,z0)

print("First odd root =", first_odd_root)

print("Function value at root =",odd_transcendental(first_odd_root, z0))

print(np.pi / 2 < first_odd_root < np.pi)

z0 = 8.100438628338855
Half-width = 5e-10 m
-6.481068870539255
[-7.81184139 -6.48106887  0.13849686]
First odd root = 2.7899695689598847
Function value at root = -6.766880389363905e-10
True


In [21]:
z0 = 8.100438628338855
even_intervals, odd_intervals = intervals(z0)

print(even_intervals)
print(odd_intervals)

[(np.float64(0.0), np.float64(1.5707963267948966)), (np.float64(3.141592653589793), np.float64(4.71238898038469)), (np.float64(6.283185307179586), np.float64(7.853981633974483))]
[(np.float64(1.5707963267948966), np.float64(3.141592653589793)), (np.float64(4.71238898038469), np.float64(6.283185307179586)), (np.float64(7.853981633974483), 8.100438628338855)]


In [28]:
even_valid_intervals, odd_valid_intervals = sign_check(
    even_intervals,
    odd_intervals,
    z0
)

print("Even valid intervals:")
print(even_valid_intervals)

print("\nOdd valid intervals:")
print(odd_valid_intervals)

Even valid intervals:
[(np.float64(1e-10), np.float64(1.5707963266948965)), (np.float64(3.141592653689793), np.float64(4.71238898028469)), (np.float64(6.283185307279586), np.float64(7.853981633874483))]

Odd valid intervals:
[(np.float64(1.5707963268948966), np.float64(3.141592653489793)), (np.float64(4.71238898048469), np.float64(6.283185307079586)), (np.float64(7.853981634074483), 8.100438628238855)]


In [36]:
even_roots, odd_roots = finding_roots(
    even_valid_intervals,
    odd_valid_intervals,
    z0
)

print("Even roots:", even_roots)
print("Odd roots:", odd_roots)

def validating_roots(even_roots,odd_roots,z0):
    f_even = even_transcendental(even_roots,z0)
    f_odd  = odd_transcendental(odd_roots,z0)

    print("Value of function at even roots : ",f_even)
    print("Value of function at odd roots : ",f_odd)

validating_roots(even_roots,odd_roots,z0)

Even roots: [np.float64(1.3974176443646402), np.float64(4.1714247066316155), np.float64(6.846938844525489)]
Odd roots: [np.float64(2.7899695689874235), np.float64(5.531507793498257), np.float64(8.006457642202896)]
Value of function at even roots :  [ 2.07167350e-09 -5.22033083e-10 -2.66523692e-10]
Value of function at odd roots :  [ 5.61639624e-11 -5.11946041e-12  1.43062229e-10]
